# Exploración de disponibilidad — V-Dem, 79 índices intermedios

Aquí evaluamos la disponibilidad real de datos: cobertura entre países, cobertura temporal y densidad
real de la matriz país-variable-año. El enfoque replica el usado en
`02_disponibilidad_datos_wdi.ipynb` para el WDI, adaptado a la estructura de V-Dem.




# Sección 1: Datos completos de V-Dem (172 países)

## Bloque 1 — Importación de librerías y carga de datos



In [35]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

df_vdem_base = pd.read_excel("df_VDEM_base.xlsx", sheet_name="df_VDEM_base")
df_metadata_vars = pd.read_excel("metadatavdem_79_indices_intermedios.xlsx", sheet_name="Sheet1")

print("df_vdem_base:", df_vdem_base.shape)
print("df_metadata_vars:", df_metadata_vars.shape)
df_vdem_base.head(3)

df_vdem_base: (10398, 88)
df_metadata_vars: (79, 4)


,Country or Area,ISO-alpha2 Code,M49_region,Region Name,M49_subregion,Sub-region Name,cow_code_countryVdem,country_name_VDEM,year,v2x_suffr,...,v2xpas_economic_opposition,v2xed_ed_poed,v2xed_ed_cent,v2xed_ed_ctag,v2xed_ed_con,v2xed_ed_dmcon,v2xed_ed_ptcon,v2xed_ptcon,v2xedvd_me_cent,v2xedvd_me_ctag
0,United States of America,US,19,Americas,21,Northern America,2,United States of America,1960,0.95,...,NaN,0.258,0.003,0.219,0.811,0.831,0.734,0.646,0.181,0.132
1,United States of America,US,19,Americas,21,Northern America,2,United States of America,1961,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132
2,United States of America,US,19,Americas,21,Northern America,2,United States of America,1962,0.95,...,NaN,0.258,0.003,0.219,0.818,0.841,0.734,0.646,0.181,0.132


## Acumuladores para el reporte


In [36]:
figuras_reporte_vdem = []
tablas_reporte_vdem = []


## Bloque 2 — Diccionario de variables (código → nombre descriptivo)

**Objetivo:** construir un diccionario que traduzca cada código técnico V-Dem
(ej. `v2x_suffr`) a su nombre descriptivo (ej. "Share of population with suffrage"),
para usarlo más adelante al etiquetar ejes y leyendas de gráficos 



In [37]:
dic_nombres_vdem = df_metadata_vars.set_index("columna")["nombre"].to_dict()

print(f"Entradas en el diccionario: {len(dic_nombres_vdem)}")
list(dic_nombres_vdem.items())[:5]

Entradas en el diccionario: 79


[('v2x_suffr', 'Share of population with suffrage'),
 ('v2x_jucon', 'Judicial constraints on the executive index'),
 ('v2xlg_legcon', 'Legislative constraints on the executive index'),
 ('v2x_cspart', 'Civil society participation index'),
 ('v2xdd_dd', 'Direct popular vote index')]

## Bloque 3 — Columnas identificatorias vs. columnas de variable

**Objetivo:** separar explícitamente las columnas identificatorias (país, región,
año, etc.) de las 79 columnas de variables V-Dem, y verificar que coinciden 1:1 con
la metadata cargada en el Bloque 2.


In [38]:
# var_cols se define a partir de las 79 columnas presentes en la metadata (fuente
# confiable), en vez de listar a mano las columnas identificatorias — así el chequeo
# es robusto ante columnas extra o con nombres distintos en el archivo fuente.
var_cols = [c for c in df_vdem_base.columns if c in dic_nombres_vdem]
id_cols = [c for c in df_vdem_base.columns if c not in dic_nombres_vdem]

print(f"Columnas identificatorias: {len(id_cols)}")
print(id_cols)
print(f"Columnas de variable (V-Dem): {len(var_cols)}")

faltan_en_df = set(dic_nombres_vdem) - set(var_cols)
assert not faltan_en_df, f"Variables de la metadata que no aparecen en df_vdem_base: {faltan_en_df}"
assert len(var_cols) == 79, f"Se esperaban 79 variables, se encontraron {len(var_cols)}"
print("OK: las 79 variables coinciden exactamente entre df_vdem_base y la metadata.")

Columnas identificatorias: 9
['Country or Area', 'ISO-alpha2 Code', 'M49_region', 'Region Name', 'M49_subregion', 'Sub-region Name', 'cow_code_countryVdem', 'country_name_VDEM', 'year']
Columnas de variable (V-Dem): 79
OK: las 79 variables coinciden exactamente entre df_vdem_base y la metadata.


## Bloque 4 — Reshape de formato ancho a formato largo



In [39]:
vdem_long = df_vdem_base.melt(
    id_vars=["cow_code_countryVdem", "Country or Area", "Region Name", "Sub-region Name", "year"],
    value_vars=var_cols,
    var_name="Variable",
    value_name="Value",
)

print(f"Filas en formato largo: {len(vdem_long)} (esperado: {len(df_vdem_base)} x {len(var_cols)} = {len(df_vdem_base) * len(var_cols)})")
print(f"Rango de años: {vdem_long['year'].min()} - {vdem_long['year'].max()}")
print(f"Países: {vdem_long['cow_code_countryVdem'].nunique()}")
print(f"Variables: {vdem_long['Variable'].nunique()}")

Filas en formato largo: 821442 (esperado: 10398 x 79 = 821442)
Rango de años: 1960 - 2024
Países: 172
Variables: 79


## Bloque 5 — Disponibilidad por variable

**Objetivo:** para cada una de las 79 variables, calcular el % de países (sobre los
172 de V-Dem) que reportan al menos un dato en algún año, y el primer/último año con
dato.

**Resultado esperado:** `disponibilidad_variable_vdem`, 79 filas.

In [40]:
n_paises_total_vdem = vdem_long["cow_code_countryVdem"].nunique()
n_variables_total = len(var_cols)

print(f"Países V-Dem: {n_paises_total_vdem}")
print(f"Variables V-Dem: {n_variables_total}")

paises_con_dato_var = (
    vdem_long.dropna(subset=["Value"])
    .groupby("Variable")["cow_code_countryVdem"]
    .nunique()
    .rename("n_paises_con_dato")
)

rango_anios_var = (
    vdem_long.dropna(subset=["Value"])
    .groupby("Variable")["year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

disponibilidad_variable_vdem = (
    pd.DataFrame(index=var_cols)
    .join(paises_con_dato_var)
    .join(rango_anios_var)
)
disponibilidad_variable_vdem["n_paises_con_dato"] = disponibilidad_variable_vdem["n_paises_con_dato"].fillna(0)
disponibilidad_variable_vdem["pct_paises_con_dato"] = disponibilidad_variable_vdem["n_paises_con_dato"] / n_paises_total_vdem
disponibilidad_variable_vdem["nombre"] = disponibilidad_variable_vdem.index.map(dic_nombres_vdem)
disponibilidad_variable_vdem["primer_anio"] = disponibilidad_variable_vdem["primer_anio"].fillna("sin_dato")
disponibilidad_variable_vdem["ultimo_anio"] = disponibilidad_variable_vdem["ultimo_anio"].fillna("sin_dato")

disponibilidad_variable_vdem.sort_values("pct_paises_con_dato").head(10)

Países V-Dem: 172
Variables V-Dem: 79


,n_paises_con_dato,primer_anio,ultimo_anio,pct_paises_con_dato,nombre
v2xed_ptcon,152,1960,2021,0.883721,Patriotic indoctrination content in education ...
v2xed_ed_dmcon,152,1960,2021,0.883721,Democratic indoctrination content in education
v2xed_ed_con,152,1960,2021,0.883721,Indoctrination content in education
v2xedvd_me_ctag,153,1960,2021,0.889535,Control over media agents
v2xed_ed_ctag,153,1960,2021,0.889535,Control over educational agents
v2xed_ed_ptcon,154,1960,2021,0.895349,Patriotic indoctrination content in education
v2xed_ed_cent,154,1960,2021,0.895349,Centralization of the education system
v2xed_ed_poed,154,1960,2021,0.895349,Political education effort in education
v2xpas_exclusion,162,1970,2019,0.941860,Party-System Exclusion Index
v2xpas_exclusion_government,162,1970,2019,0.941860,Government Coalition Exclusion Index


**Qué comprobar:** `disponibilidad_variable_vdem.shape[0] == 79`; que
`pct_paises_con_dato` esté entre 0 y 1; revisar si aparece algún `"sin_dato"` (variable
sin ningún dato para ningún país — sería una señal de alerta a investigar antes de
seguir, no algo esperable dado que estas 79 ya fueron pre-seleccionadas).

## Bloque 6 — Disponibilidad por país

**Objetivo:** para cada uno de los 172 países, calcular cuántas de las 79 variables
tiene con al menos un dato, y el primer/último año con dato.

**Resultado esperado:** `disponibilidad_pais_vdem`, 172 filas.

In [41]:
variables_con_dato_pais = (
    vdem_long.dropna(subset=["Value"])
    .groupby("cow_code_countryVdem")["Variable"]
    .nunique()
    .rename("n_variables_con_dato")
)

rango_anios_pais = (
    vdem_long.dropna(subset=["Value"])
    .groupby("cow_code_countryVdem")["year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

info_pais = (
    df_vdem_base[["cow_code_countryVdem", "Country or Area", "Region Name", "Sub-region Name"]]
    .drop_duplicates("cow_code_countryVdem")
    .set_index("cow_code_countryVdem")
)

disponibilidad_pais_vdem = (
    info_pais
    .join(variables_con_dato_pais)
    .join(rango_anios_pais)
)
disponibilidad_pais_vdem["n_variables_con_dato"] = disponibilidad_pais_vdem["n_variables_con_dato"].fillna(0)
disponibilidad_pais_vdem["pct_variables_con_dato"] = disponibilidad_pais_vdem["n_variables_con_dato"] / n_variables_total
disponibilidad_pais_vdem["primer_anio"] = disponibilidad_pais_vdem["primer_anio"].fillna("sin_dato")
disponibilidad_pais_vdem["ultimo_anio"] = disponibilidad_pais_vdem["ultimo_anio"].fillna("sin_dato")

disponibilidad_pais_vdem.sort_values("pct_variables_con_dato").head(10)

,Country or Area,Region Name,Sub-region Name,n_variables_con_dato,primer_anio,ultimo_anio,pct_variables_con_dato
cow_code_countryVdem,,,,,,,
698,Oman,Asia,Western Asia,64,1960,2024,0.810127
940,Solomon Islands,Oceania,Melanesia,64,1960,2024,0.810127
626,South Sudan,Africa,Sub-Saharan Africa,67,2011,2024,0.848101
694,Qatar,Asia,Western Asia,67,1960,2024,0.848101
670,Saudi Arabia,Asia,Western Asia,67,1960,2024,0.848101
115,Suriname,Americas,Latin America and the Caribbean,68,1960,2024,0.860759
860,Timor-Leste,Asia,South-eastern Asia,70,1960,2024,0.886076
42,Dominican Republic,Americas,Latin America and the Caribbean,71,1960,2024,0.898734
403,Sao Tome and Principe,Africa,Sub-Saharan Africa,71,1960,2024,0.898734


## Bloque 7 — Disponibilidad por año

**Objetivo:** calcular, para cada año del panel (1960–2024), cuántos países reportan
al menos un dato y qué proporción representa; además, la **densidad real** de la
matriz país-variable.


pct_paises_con_dato : Nos dice para cada año, cuenta cuántos de los 172 países tienen al menos un dato en al menos una de las 79 variables ese año

densidad_real : "de todo lo que podría haber, ¿cuánto hay realmente?" para cada año, el máximo posible de datos es 172 países × 79 variables = 13.588 casilleros. densidad_real cuenta cuántos de esos casilleros específicos tienen un valor cargado, y calcula la proporción.


**Resultado esperado:** `disponibilidad_anio_vdem`, una fila por año.

In [42]:
anios_totales_vdem = sorted(vdem_long["year"].unique())

paises_con_dato_anio = (
    vdem_long.dropna(subset=["Value"])
    .groupby("year")["cow_code_countryVdem"]
    .nunique()
    .rename("n_paises_con_dato")
)

disponibilidad_anio_vdem = pd.DataFrame(index=anios_totales_vdem).join(paises_con_dato_anio)
disponibilidad_anio_vdem["n_paises_con_dato"] = disponibilidad_anio_vdem["n_paises_con_dato"].fillna(0)
disponibilidad_anio_vdem["pct_paises_con_dato"] = disponibilidad_anio_vdem["n_paises_con_dato"] / n_paises_total_vdem

# Densidad real: % de celdas país-variable efectivamente completas en cada año
celdas_posibles_anio = n_paises_total_vdem * n_variables_total
celdas_con_dato_anio = vdem_long.dropna(subset=["Value"]).groupby("year").size()
disponibilidad_anio_vdem["densidad_real"] = (
    celdas_con_dato_anio.reindex(anios_totales_vdem, fill_value=0).values / celdas_posibles_anio
)

disponibilidad_anio_vdem.tail(15)

,n_paises_con_dato,pct_paises_con_dato,densidad_real
2010,171,0.994186,0.858478
2011,172,1.000000,0.870327
2012,172,1.000000,0.863262
2013,172,1.000000,0.867457
2014,172,1.000000,0.867015
2015,172,1.000000,0.866058
2016,172,1.000000,0.866721
2017,172,1.000000,0.858846
2018,172,1.000000,0.859656
2019,172,1.000000,0.867825


Hasta aquí hemos visto que la cobertura de datos está bastante completa, superando en todos los casos el 84%. 

## Bloque 8 — Gráficos de disponibilidad

A partir de acá se generan las visualizaciones exploratorias.

### Gráfico 1 — Cobertura por variable (histograma)

**Objetivo:** ver cómo se distribuye, entre las 79 variables, el % de países que las
reportan.

In [43]:
df_plot_var = disponibilidad_variable_vdem.reset_index().rename(columns={"index": "Variable"})

fig_cobertura_variable = px.histogram(
    df_plot_var,
    x="pct_paises_con_dato",
    nbins=30,
    hover_data=["Variable", "nombre", "n_paises_con_dato", "primer_anio", "ultimo_anio"],
    labels={"pct_paises_con_dato": "% de países V-Dem con al menos un dato"},
)
fig_cobertura_variable.update_layout(
    title="Cobertura por variable: ¿a cuántos países alcanza cada índice V-Dem?",
    xaxis_title="% de países con al menos un dato",
    yaxis_title="Cantidad de variables",
    bargap=0.05,
    margin=dict(b=100),
)
fig_cobertura_variable.add_annotation(
    text="Nota: cada barra agrupa variables según el % de los 172 países V-Dem que las reportan (al menos un dato, en algún año).",
    xref="paper", yref="paper", x=0, y=-0.22, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("cobertura_variable", fig_cobertura_variable))
fig_cobertura_variable.show()


### Gráfico 2 — Cobertura por país (barras horizontales, coloreado por región)

**Objetivo:** ver qué proporción de las 79 variables reporta cada país, ordenado de
menor a mayor cobertura.

In [44]:
df_plot_pais = disponibilidad_pais_vdem.reset_index().sort_values("pct_variables_con_dato")

fig_cobertura_pais = px.bar(
    df_plot_pais,
    x="pct_variables_con_dato",
    y="Country or Area",
    color="Region Name",
    orientation="h",
    hover_data=["n_variables_con_dato", "primer_anio", "ultimo_anio", "Sub-region Name"],
    labels={"pct_variables_con_dato": "% de variables con al menos un dato"},
)
fig_cobertura_pais.update_layout(
    title="Cobertura por país: ¿qué proporción de las 79 variables V-Dem reporta cada país?",
    xaxis_title="% de variables con al menos un dato",
    yaxis_title="País",
    height=2600,
    margin=dict(b=90),
)
# Nota como anotación chica al pie (con height=2600 una anotación grande queda muy lejos del eje;
# se agrega igual como referencia textual, aunque en un gráfico tan alto conviene leer la nota
# desde este bloque de código más que desde el gráfico en sí).
fig_cobertura_pais.add_annotation(
    text="Nota: % de las 79 variables candidatas con al menos un dato para cada país, coloreado por región M49.",
    xref="paper", yref="paper", x=0, y=-0.03, showarrow=False,
    font=dict(size=11, color="gray"), align="left",
)

figuras_reporte_vdem.append(("cobertura_pais", fig_cobertura_pais))
fig_cobertura_pais.show()


### Gráfico 3 — Dispersión de cobertura por región

**Objetivo:** ver si hay regiones sistemáticamente peor cubiertas en V-Dem, igual
chequeo que se hizo para el WDI.

In [45]:
fig_dispersion_region = px.box(
    df_plot_pais,
    x="Region Name",
    y="pct_variables_con_dato",
    points="all",
    hover_data=["Country or Area", "n_variables_con_dato", "Sub-region Name"],
    labels={"pct_variables_con_dato": "% de variables con al menos un dato"},
)
fig_dispersion_region.update_layout(
    title="Dispersión de cobertura por región (V-Dem): ¿hay regiones sistemáticamente peor cubiertas?",
    xaxis_title="Región",
    yaxis_title="% de variables con al menos un dato",
    margin=dict(b=110),
)
fig_dispersion_region.add_annotation(
    text="Nota: cada punto es un país; la caja resume la dispersión de cobertura entre los países de esa región.",
    xref="paper", yref="paper", x=0, y=-0.25, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("dispersion_region", fig_dispersion_region))
fig_dispersion_region.show()


Los continentes tienen datos bastante concentrados, como veníamos viendo en las visualizaciones anteriores. Oceanía es, al igual que en la base WDI el continente con menos registros y más disperso (aunque con mucha menos diferencia que en la base WDI)

# Sección 2: Cobertura respecto al universo ONU

La fuente V-Dem se construye sobre el sistema de códigos históricos COW (Correlates of War), que incluye tanto códigos vigentes como códigos de continuidad histórica de estados ya desaparecidos. Se ha estandarizado con sistema ONU y se ha filtrado por mimebros ONU. 

Acá vamos a ver dos cosas

- **Dirección 1 — entidades de V-Dem que no son miembros ONU actuales:** 

Hay códigos de Vdem que NO estan en ONU. Esto sucede por dos causas.
1. Porque los países cambian sus denominaciones a lo largo del tiempo. A través de un chequeo manual, se realizaron recodificaciones
para incluir a esos países

2. Porque sencillamente no son miembros de ONU. Entonces al filtrar, se pierden. Esta perdida se ha dejado documentada por si en un futuro
se quiere trabajar este aspecto en mayor profundidad. 

- **Dirección 2 — miembros ONU sin ningún código COW en V-Dem:** 
Hay mimebros de ONU que no son considerados por VDEM. Éstos representan una ausencia real del dato en la fuente. 
Esto también está documentado en detalle. 




**Objetivo:** Dejar documentado los países o territorios que no entraran al análisis. 

Los resultados de esta sección no estarán disponibles en el HTML (aquí solo se menciona lo realizado), pero sí en el excel publicado junto al repositorio.

**Resultado esperado:** `df_regiones_onu_vdem` con 193 filas.

In [46]:
df_regiones_onu_vdem = pd.read_excel("df_regiones_miembros_onu.xlsx")
print("df_regiones_onu_vdem:", df_regiones_onu_vdem.shape)


df_regiones_onu_vdem: (193, 13)


**Objetivo:** cargar la lista completa de países/códigos COW tal como los trae V-Dem *antes* del inner join con ONU (`df_paises_vdem.xlsx`, generado en el notebook 06) — es la pieza que faltaba para poder documentar la Dirección 1.

**Resultado esperado:** `df_paises_vdem_raw`, 211 filas (`country_name`, `COWcode`; algunos países aparecen más de una vez por continuidad histórica de códigos).

In [47]:
df_paises_vdem_raw = pd.read_excel("df_paises_vdem.xlsx")
print("df_paises_vdem_raw:", df_paises_vdem_raw.shape)
df_paises_vdem_raw.head()


df_paises_vdem_raw: (211, 4)


,country_name,country_text_id,country_id,COWcode
0,Afghanistan,AFG,36,700.0
1,Albania,ALB,12,339.0
2,Algeria,DZA,103,615.0
3,Angola,AGO,104,540.0
4,Argentina,ARG,37,160.0


**Objetivo:** identificar qué códigos COW de la fuente V-Dem no corresponden a ningún miembro ONU vigente (Dirección 1).

**Resultado esperado:** `df_vdem_sin_match_onu`, 24 filas.

In [48]:
codigos_cow_onu_vigentes = set(df_regiones_onu_vdem["cow_code_countryVdem"].dropna())
codigos_cow_vdem_raw = set(df_paises_vdem_raw["COWcode"].dropna())

codigos_vdem_no_onu = codigos_cow_vdem_raw - codigos_cow_onu_vigentes

df_vdem_sin_match_onu = (
    df_paises_vdem_raw[df_paises_vdem_raw["COWcode"].isin(codigos_vdem_no_onu)]
    [["country_name", "COWcode"]]
    .sort_values("COWcode")
    .reset_index(drop=True)
)

print(f"Códigos únicos en V-Dem (raw): {len(codigos_cow_vdem_raw)}")
print(f"Códigos de los 193 miembros ONU: {len(codigos_cow_onu_vigentes)}")
print(f"[Dirección 1] Entidades V-Dem que no matchean con ningún miembro ONU vigente: {len(df_vdem_sin_match_onu)}")
df_vdem_sin_match_onu


Códigos únicos en V-Dem (raw): 196
Códigos de los 193 miembros ONU: 193
[Dirección 1] Entidades V-Dem que no matchean con ningún miembro ONU vigente: 24


,country_name,COWcode
0,Hanover,240.0
1,Bavaria,245.0
2,Germany,260.0
3,German Democratic Republic,265.0
4,Baden,267.0
5,Saxony,269.0
6,Würtemberg,271.0
7,Hesse-Kassel,273.0
8,Hesse-Darmstadt,275.0
9,Mecklenburg Schwerin,280.0


**Objetivo:** identificar, por código COW, qué miembros ONU no tienen ninguna fila en `df_vdem_base` (Dirección 2).

**Resultado esperado:** `df_onu_sin_vdem`, 21 filas (según lo ya documentado en el notebook 06).

In [49]:
codigos_cow_onu = set(df_regiones_onu_vdem["cow_code_countryVdem"].dropna())
codigos_cow_vdem = set(df_vdem_base["cow_code_countryVdem"].dropna())

paises_onu_sin_vdem = codigos_cow_onu - codigos_cow_vdem

df_onu_sin_vdem = (
    df_regiones_onu_vdem[df_regiones_onu_vdem["cow_code_countryVdem"].isin(paises_onu_sin_vdem)]
    [["Country or Area", "cow_code_countryVdem", "Region Name"]]
    .sort_values("Region Name")
    .reset_index(drop=True)
)

print(f"Países ONU ausentes de V-Dem: {len(df_onu_sin_vdem)} de 193")
df_onu_sin_vdem


Países ONU ausentes de V-Dem: 21 de 193


,Country or Area,cow_code_countryVdem,Region Name
0,Saint Lucia,56,Americas
1,Saint Kitts and Nevis,60,Americas
2,Grenada,55,Americas
3,Saint Vincent and the Grenadines,57,Americas
4,Bahamas,31,Americas
5,Antigua and Barbuda,58,Americas
6,Dominica,54,Americas
7,Belize,80,Americas
8,Brunei Darussalam,835,Asia
9,Andorra,232,Europe


Vamos a ver una lista de valores unicos de paises que sí encontraron match

In [50]:
codigos_vdem_match_onu = codigos_cow_vdem_raw & codigos_cow_onu_vigentes

df_vdem_match_onu = (
    df_paises_vdem_raw[df_paises_vdem_raw["COWcode"].isin(codigos_vdem_match_onu)]
    [["country_name", "COWcode"]]
    .merge(
        df_regiones_onu_vdem[["Member State", "cow_code_countryVdem"]],
        left_on="COWcode", right_on="cow_code_countryVdem", how="left"
    )
    .drop(columns="cow_code_countryVdem")
    .sort_values("COWcode")
    .reset_index(drop=True)
)

print(f"Entidades V-Dem que matchean con ONU: {len(df_vdem_match_onu)} / 193")
df_vdem_match_onu

Entidades V-Dem que matchean con ONU: 172 / 193


,country_name,COWcode,Member State
0,United States of America,2.0,United States
1,Canada,20.0,Canada
2,Cuba,40.0,Cuba
3,Haiti,41.0,Haiti
4,Dominican Republic,42.0,Dominican Republic
...,...,...,...
167,Papua New Guinea,910.0,Papua New Guinea
168,New Zealand,920.0,New Zealand
169,Vanuatu,935.0,Vanuatu
170,Solomon Islands,940.0,Solomon Islands


In [51]:
df_onu_sin_vdem["Region Name"].value_counts()


Region Name
Americas    8
Oceania     8
Europe      4
Asia        1
Name: count, dtype: int64

# Sección 3: Análisis de recorte temporal

### Gráfico 4 — Densidad real de la matriz país-variable a lo largo del tiempo

**Objetivo:** identificar a partir de qué año la matriz país-variable está
razonablemente completa (insumo empírico para la decisión pendiente sobre el año de
inicio del panel).

In [52]:
fig_densidad_anio = px.line(
    disponibilidad_anio_vdem.reset_index().rename(columns={"index": "year"}),
    x="year",
    y="densidad_real",
    markers=True,
    labels={"densidad_real": "% de celdas país-variable con dato"},
)
fig_densidad_anio.update_layout(
    title=dict(text="% de cobertura de datos a lo largo del tiempo (VDEM)", x=0.5, xanchor="center"),
    xaxis_title="Año",
    yaxis_title="% de celdas país-variable con dato",
    yaxis_range=[0, 1],
    margin=dict(b=90),
)
fig_densidad_anio.add_annotation(
    text="Nota: % de la matriz países\u00d7indicadores que est\u00e1 completa en cada a\u00f1o.",
    xref="paper", yref="paper", x=0, y=-0.22, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("densidad_anio", fig_densidad_anio))
fig_densidad_anio.show()

Vemos una importante caída luego del 2020. A continuación exploramos un poco esta caída:

In [53]:
variables_corte_temprano = (
    disponibilidad_variable_vdem[disponibilidad_variable_vdem["ultimo_anio"] < 2024]
    .sort_values("ultimo_anio")
    [["nombre", "ultimo_anio"]]
)

print(f"Variables con corte temprano: {len(variables_corte_temprano)} de {len(disponibilidad_variable_vdem)}")
variables_corte_temprano.groupby("ultimo_anio").size().rename("n_variables")

Variables con corte temprano: 26 de 79


ultimo_anio
2019    12
2021     9
2023     5
Name: n_variables, dtype: int64

In [54]:
detalle_corte_temprano = (
    disponibilidad_variable_vdem[disponibilidad_variable_vdem["ultimo_anio"] < 2024]
    .reset_index()
    .rename(columns={"index": "codigo"})
    [["codigo", "nombre", "ultimo_anio"]]
    .sort_values(["ultimo_anio", "codigo"])
    .reset_index(drop=True)
)

print(f"Variables con corte temprano: {len(detalle_corte_temprano)} de {len(disponibilidad_variable_vdem)}")
detalle_corte_temprano

Variables con corte temprano: 26 de 79


,codigo,nombre,ultimo_anio
0,v2xpas_democracy,Party-System Democracy Index,2019
1,v2xpas_democracy_government,Government Coalition Democracy Index,2019
2,v2xpas_democracy_opposition,Opposition Parties' Democracy Index,2019
3,v2xpas_economic,Party-System Left-Right Index,2019
4,v2xpas_economic_government,Government Coalition Left-Right Index,2019
5,v2xpas_economic_opposition,Opposition Parties Left-Right Index,2019
6,v2xpas_exclusion,Party-System Exclusion Index,2019
7,v2xpas_exclusion_government,Government Coalition Exclusion Index,2019
8,v2xpas_exclusion_opposition,Opposition Parties' Exclusion Index,2019
9,v2xpas_religion,Party-System Religion Index,2019


Revisando cuales son las variables con corte temprano y la sincronicidad con las que faltan (por ejemplo muchas de ellas coinciden en su ausencia a partir del mismo año) y lo leído en el coodebook, creemos que esta bajada es esperable dada la naturaleza de los datos. 

12 variables de sistema de partidos (v2xpas_*: democracia, religión, exclusión y posición económica de partidos, gobierno y oposición) — última observación en 2019. Según el codebook de V-Dem, estos índices se calculan a nivel país-año-electoral, y no siguen actualizándose año a año fuera de ese ciclo.

9 variables de indoctrinación educativa y mediática (v2xed_*, v2xedvd_*) — última observación en 2021. Provienen del módulo Varieties of Indoctrination (V-Indoc), un relevamiento de expertos independiente que cubrió el período 1945–2021 y no forma parte de la actualización anual regular de V-Dem.

5 variables de exclusión por grupo (v2xpe_exl*: género, socioeconómico, político, geográfico, socioeconómico) — última observación en 2023, un año de rezago respecto al resto del panel.

### Gráfico 5 — Heatmap de disponibilidad regional en el tiempo

**Objetivo:** ver si la densidad de datos mejora de forma pareja entre regiones a lo
largo del tiempo, o si hay regiones sistemáticamente rezagadas (mismo tipo de
hallazgo que se encontró para Oceanía en el WDI).

In [55]:
vdem_long_con_dato = vdem_long.dropna(subset=["Value"])

n_paises_por_region = df_vdem_base.drop_duplicates("cow_code_countryVdem").groupby("Region Name")["cow_code_countryVdem"].nunique()
celdas_posibles_region = n_paises_por_region * n_variables_total

celdas_con_dato_region_anio = (
    vdem_long_con_dato.groupby(["year", "Region Name"]).size().unstack("Region Name")
)
densidad_region_anio = celdas_con_dato_region_anio.div(celdas_posibles_region, axis=1).fillna(0)
matriz_region_anio = densidad_region_anio.T

fig_heatmap_region = px.imshow(
    matriz_region_anio,
    labels=dict(x="Año", y="Región", color="% celdas con dato"),
    x=matriz_region_anio.columns,
    y=matriz_region_anio.index,
    color_continuous_scale="Blues",
    aspect="auto",
)
fig_heatmap_region.update_layout(
    title="Disponibilidad regional de V-Dem a lo largo del tiempo",
    margin=dict(b=90),
)
fig_heatmap_region.add_annotation(
    text="Nota: % de celdas país-variable con dato dentro de cada región, para cada año.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("heatmap_region_anio", fig_heatmap_region))
fig_heatmap_region.show()


Acá podemos ver que hay continentes que por años tuvieron bajo registro de datos. Pero desde 1995 estas diferencias se estabilizan

In [56]:
fig_lineas_region_anio = px.line(
    densidad_region_anio.reset_index(),
    x="year",
    y=densidad_region_anio.columns.tolist(),
    labels={"year": "Año", "value": "% de celdas país-variable con dato", "variable": "Región"},
)
fig_lineas_region_anio.update_layout(
    title="Disponibilidad regional de V-Dem a lo largo del tiempo (líneas)",
    yaxis_tickformat=".0%",
    legend_title_text="Región",
    margin=dict(b=90),
)
fig_lineas_region_anio.add_annotation(
    text="Nota: mismos datos que el heatmap de arriba — % de celdas país-variable con dato dentro de cada región, para cada año.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("lineas_region_anio", fig_lineas_region_anio))
fig_lineas_region_anio.show()

Estos datos tienen bastante sentido.

Asia
Tiene el salto de 63% a 69% en 1970 y el segundo salto grande en 1990. Coincide con los movimientos de la URSS

Europa
el salto de 55% a 90% justo en 1989-1990 coincide con caída de URSS y también con la disolución de la ex-yugoslavia

### Gráfico 6 — Dispersión de la densidad real por década

**Objetivo:** ver si la densidad de datos fue estable año a año dentro de cada década, o si hubo mucha variación interna — mismo chequeo que se hizo para el WDI.

In [57]:
df_anio_plot_vdem = disponibilidad_anio_vdem.reset_index().rename(columns={"index": "year"})
df_anio_plot_vdem["decada"] = (df_anio_plot_vdem["year"] // 10 * 10).astype(str) + "s"

fig_dispersion_decada_vdem = px.box(
    df_anio_plot_vdem,
    x="decada",
    y="densidad_real",
    points="all",
    labels={"densidad_real": "% de celdas país-variable con dato", "decada": "Década"},
)
fig_dispersion_decada_vdem.update_layout(
    title="Estabilidad de la densidad real por década (V-Dem)",
    xaxis_title="Década",
    yaxis_title="% de celdas país-variable con dato",
    margin=dict(b=90),
)
fig_dispersion_decada_vdem.add_annotation(
    text="Nota: cada punto es un año dentro de la década; la caja resume la dispersión de la densidad real entre esos años.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("dispersion_decada_vdem", fig_dispersion_decada_vdem))
fig_dispersion_decada_vdem.show()


Vemos que la caja mas dispersa es la de la ultima decada. Esto es esperable dado el delay de reporte por parte de los países, y también es consistente por el tipo de indicadores que antes habíamos visto que no estaban reportados en los últimos años: responden a ciclos electorales o que no son de reporte anual, entonces es esperable que aún no esten construidos

### Gráfico 7 — Curva de sensibilidad de la densidad al año de inicio del panel

**Objetivo:** igual que en el WDI, evaluar empíricamente el trade-off entre extender el panel hacia atrás en el tiempo y perder completitud, para el universo V-Dem (172 países, 79 variables). Sirve como insumo empírico adicional para la decisión sobre el año de inicio del panel (1990 vs. alternativas).

In [58]:
anio_min_disponible_vdem = int(vdem_long["year"].min())
anio_max_disponible_vdem = int(vdem_long["year"].max())

anios_inicio_candidatos_vdem = list(range(
    (anio_min_disponible_vdem // 5) * 5, anio_max_disponible_vdem + 1, 5
))

def calcular_densidad_panel_vdem(datos_vdem, anio_inicio, anio_fin):
    sub = datos_vdem[(datos_vdem["year"] >= anio_inicio) & (datos_vdem["year"] <= anio_fin)]
    n_celdas_totales = n_paises_total_vdem * n_variables_total * (anio_fin - anio_inicio + 1)
    n_celdas_con_dato = sub["Value"].notna().sum()
    return n_celdas_con_dato / n_celdas_totales

densidad_por_inicio_vdem = [
    calcular_densidad_panel_vdem(vdem_long, anio_inicio, anio_max_disponible_vdem)
    for anio_inicio in anios_inicio_candidatos_vdem
]

fig_curva_sensibilidad_vdem = px.line(
    x=anios_inicio_candidatos_vdem, y=densidad_por_inicio_vdem, markers=True,
    labels={"x": "Año de inicio candidato", "y": "Densidad del panel"},
)
fig_curva_sensibilidad_vdem.add_vline(x=1990, line_dash="dash", line_color="gray")
fig_curva_sensibilidad_vdem.update_layout(
    title="Sensibilidad de la densidad del panel al año de inicio (V-Dem)",
    yaxis_tickformat=".0%",
    margin=dict(b=90),
)
fig_curva_sensibilidad_vdem.add_annotation(
    text="Nota: cada punto es la densidad de todo el panel (172×79×años) si arrancara en ese año, hasta el último año disponible.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("curva_sensibilidad_anio_inicio_vdem", fig_curva_sensibilidad_vdem))
fig_curva_sensibilidad_vdem.show()


# Sección 4: Análisis de dispersión entre escenarios

**Adaptación respecto al WDI:** en el notebook 02 comparábamos tres universos de países (WDI completo, ONU, ONU 1990-2024), porque el filtro de países cambiaba el denominador. Acá V-Dem ya viene fijo en 172 países (no hay "WDI vs. ONU" que filtrar — eso ya se resolvió, y de forma distinta, en la Sección 2). La comparación relevante entonces es otra: **todo el rango temporal disponible vs. el recorte sustantivo 1990-2024**, sobre el mismo universo de 172 países. Por eso acá va a haber dos escenarios en las tablas y boxplots, no tres.

## 4.1 Dispersión de cobertura por país — variable reportada al menos una vez

In [59]:
vdem_long_9024 = vdem_long[(vdem_long["year"] >= 1990) & (vdem_long["year"] <= 2024)]

variables_con_dato_pais_9024 = (
    vdem_long_9024.dropna(subset=["Value"])
    .groupby("cow_code_countryVdem")["Variable"]
    .nunique()
    .rename("n_variables_con_dato_9024")
)

disponibilidad_pais_vdem_9024 = pd.DataFrame(index=disponibilidad_pais_vdem.index).join(variables_con_dato_pais_9024)
disponibilidad_pais_vdem_9024["n_variables_con_dato_9024"] = disponibilidad_pais_vdem_9024["n_variables_con_dato_9024"].fillna(0)
disponibilidad_pais_vdem_9024["pct_variables_con_dato_9024"] = disponibilidad_pais_vdem_9024["n_variables_con_dato_9024"] / n_variables_total

disponibilidad_pais_vdem_9024.head()


,n_variables_con_dato_9024,pct_variables_con_dato_9024
cow_code_countryVdem,,
2,79,1.0
900,79,1.0
522,79,1.0
452,79,1.0
630,79,1.0


In [60]:
comparacion_dispersion_pais_vdem = pd.DataFrame({
    "V-Dem completo (todo el rango)": disponibilidad_pais_vdem["pct_variables_con_dato"].describe(),
    "V-Dem 1990-2024": disponibilidad_pais_vdem_9024["pct_variables_con_dato_9024"].describe(),
})
comparacion_dispersion_pais_vdem.loc["IQR"] = comparacion_dispersion_pais_vdem.loc["75%"] - comparacion_dispersion_pais_vdem.loc["25%"]
comparacion_dispersion_pais_vdem.loc["CV"] = comparacion_dispersion_pais_vdem.loc["std"] / comparacion_dispersion_pais_vdem.loc["mean"]

tablas_reporte_vdem.append(("dispersion_pais_presencia_vdem", comparacion_dispersion_pais_vdem))

comparacion_dispersion_pais_vdem


,V-Dem completo (todo el rango),V-Dem 1990-2024
count,172.000000,172.000000
mean,0.980718,0.979688
std,0.042381,0.043647
min,0.810127,0.810127
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,1.000000,1.000000
IQR,0.000000,0.000000
CV,0.043215,0.044552


In [61]:
df_dispersion_presencia_vdem = pd.concat([
    pd.DataFrame({"pct_cobertura": disponibilidad_pais_vdem["pct_variables_con_dato"], "Escenario": "V-Dem completo"}),
    pd.DataFrame({"pct_cobertura": disponibilidad_pais_vdem_9024["pct_variables_con_dato_9024"], "Escenario": "V-Dem 1990-2024"}),
])

fig_dispersion_presencia_vdem = px.box(
    df_dispersion_presencia_vdem, x="Escenario", y="pct_cobertura", points="outliers",
    labels={"pct_cobertura": "% de variables con al menos un dato"},
)
fig_dispersion_presencia_vdem.update_layout(
    title="Dispersión de cobertura por país (presencia acumulada): rango completo vs. 1990-2024",
    margin=dict(b=90),
)
fig_dispersion_presencia_vdem.add_annotation(
    text="Nota: para cada país, % de las 79 variables con al menos un dato en el rango de años correspondiente.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("dispersion_pais_presencia_vdem", fig_dispersion_presencia_vdem))
fig_dispersion_presencia_vdem.show()


## 4.2 Dispersión de cobertura por país — densidad ponderada por año

Acá cada celda variable-año cuenta individualmente (no solo si la variable aparece alguna vez) — misma lógica y misma razón que en el notebook 02: es la métrica correcta para evaluar el efecto real de recortar a 1990-2024.

In [62]:
def calcular_densidad_por_pais_vdem(datos_vdem, anio_inicio, anio_fin):
    sub = datos_vdem[(datos_vdem["year"] >= anio_inicio) & (datos_vdem["year"] <= anio_fin)]
    n_celdas_por_pais = n_variables_total * (anio_fin - anio_inicio + 1)
    celdas_con_dato_por_pais = (
        sub.dropna(subset=["Value"])
        .groupby("cow_code_countryVdem")
        .size()
    )
    densidad_pais = (
        pd.Series(index=disponibilidad_pais_vdem.index, dtype=float)
        .fillna(0)
        .add(celdas_con_dato_por_pais.reindex(disponibilidad_pais_vdem.index).fillna(0), fill_value=0)
        / n_celdas_por_pais
    )
    return densidad_pais

densidad_pais_vdem_completo = calcular_densidad_por_pais_vdem(vdem_long, anio_min_disponible_vdem, anio_max_disponible_vdem)
densidad_pais_vdem_9024 = calcular_densidad_por_pais_vdem(vdem_long, 1990, 2024)

comparacion_densidad_pais_vdem = pd.DataFrame({
    "V-Dem completo (todo el rango)": densidad_pais_vdem_completo.describe(),
    "V-Dem 1990-2024": densidad_pais_vdem_9024.describe(),
})
comparacion_densidad_pais_vdem.loc["IQR"] = comparacion_densidad_pais_vdem.loc["75%"] - comparacion_densidad_pais_vdem.loc["25%"]
comparacion_densidad_pais_vdem.loc["CV"] = comparacion_densidad_pais_vdem.loc["std"] / comparacion_densidad_pais_vdem.loc["mean"]

tablas_reporte_vdem.append(("dispersion_pais_densidad_vdem", comparacion_densidad_pais_vdem))

comparacion_densidad_pais_vdem


,V-Dem completo (todo el rango),V-Dem 1990-2024
count,172.000000,172.000000
mean,0.787063,0.843482
std,0.146929,0.056347
min,0.170789,0.317179
25%,0.771178,0.843400
50%,0.857741,0.862568
75%,0.868160,0.867541
max,0.900292,0.901627
IQR,0.096981,0.024141
CV,0.186680,0.066803


In [63]:
df_dispersion_densidad_vdem = pd.concat([
    pd.DataFrame({"densidad": densidad_pais_vdem_completo, "Escenario": "V-Dem completo"}),
    pd.DataFrame({"densidad": densidad_pais_vdem_9024, "Escenario": "V-Dem 1990-2024"}),
])

fig_dispersion_densidad_vdem = px.box(
    df_dispersion_densidad_vdem, x="Escenario", y="densidad", points="outliers",
    labels={"densidad": "% de celdas variable-año con dato"},
)
fig_dispersion_densidad_vdem.update_layout(
    title="Dispersión de cobertura por país (densidad ponderada por año): rango completo vs. 1990-2024",
    margin=dict(b=90),
)
fig_dispersion_densidad_vdem.add_annotation(
    text="Nota: para cada país, % de celdas variable-año efectivamente con dato en el rango correspondiente.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("dispersion_pais_densidad_vdem", fig_dispersion_densidad_vdem))
fig_dispersion_densidad_vdem.show()


## 4.3 Oceanía: ¿qué países explican la brecha regional?

Ya vimos en el Gráfico 3 que Oceanía es la región con menor cobertura y mayor dispersión en V-Dem. Pero hay un hallazgo previo, de la Sección 2, que contextualiza esto de forma más severa que en el WDI: **8 de los 14 miembros ONU de Oceanía directamente no están en V-Dem** (Kiribati, Samoa, Nauru, Palau, Tuvalu, Tonga, Islas Marshall y Micronesia). En el WDI esos mismos países sí aparecían, aunque con baja cobertura — acá ni siquiera tienen una fila en la base. El gráfico de abajo, entonces, solo puede mostrar los 6 países de Oceanía que sí están en V-Dem.

In [64]:
paises_oceania_vdem = disponibilidad_pais_vdem[disponibilidad_pais_vdem["Region Name"] == "Oceania"].copy()
paises_oceania_vdem = paises_oceania_vdem.sort_values("pct_variables_con_dato", ascending=False)

fig_cobertura_oceania_vdem = px.bar(
    paises_oceania_vdem.reset_index(),
    x="Country or Area", y="pct_variables_con_dato",
    labels={"pct_variables_con_dato": "% de variables con al menos un dato", "Country or Area": "País"},
)
fig_cobertura_oceania_vdem.update_layout(
    title="Cobertura por país — Oceanía en V-Dem (6 de 14 miembros ONU presentes)",
    yaxis_tickformat=".0%",
    margin=dict(b=90),
)
fig_cobertura_oceania_vdem.add_annotation(
    text="Nota: los 8 miembros ONU de Oceanía ausentes de V-Dem no aparecen acá (no tienen ninguna fila en la base).",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte_vdem.append(("cobertura_oceania_vdem", fig_cobertura_oceania_vdem))
fig_cobertura_oceania_vdem.show()


# Bloque de exportación

Todo lo de acá abajo está comentado por defecto — descomentar solo los bloques que se necesiten antes de correr.

In [65]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path_vdem = OUT_DIR / "disponibilidad_vdem_79.xlsx"

with pd.ExcelWriter(out_path_vdem) as writer:
    disponibilidad_variable_vdem.to_excel(writer, sheet_name="por_variable")
    disponibilidad_pais_vdem.to_excel(writer, sheet_name="por_pais")
    disponibilidad_anio_vdem.to_excel(writer, sheet_name="por_anio")
    df_onu_sin_vdem.to_excel(writer, sheet_name="match_onu_sin_vdem", index=False)

print(f"Exportado: {out_path_vdem}")


Exportado: data\processed\disponibilidad_vdem_79.xlsx


In [66]:
with pd.ExcelWriter(out_path_vdem, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
    comparacion_dispersion_pais_vdem.to_excel(writer, sheet_name="dispersion_pais_presencia")
    comparacion_densidad_pais_vdem.to_excel(writer, sheet_name="dispersion_pais_densidad")

print(f"Hojas de dispersión añadidas a: {out_path_vdem}")


Hojas de dispersión añadidas a: data\processed\disponibilidad_vdem_79.xlsx


In [67]:
import re
import nbformat

NOTEBOOK_PATH = "07_exploracion_VDEM_Base79.ipynb"  # ajustar si el nombre/ruta difiere

figuras_por_nombre = dict(figuras_reporte_vdem)
tablas_por_nombre = dict(tablas_reporte_vdem)

def markdown_a_html(texto):
    """Conversor liviano de Markdown a HTML (headers, negrita, código inline, listas, párrafos)."""
    lineas = texto.split("\n")
    html = []
    en_lista = False
    for linea in lineas:
        l = linea.rstrip()
        if l.startswith("### "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h3>{l[4:]}</h3>")
        elif l.startswith("## "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h2>{l[3:]}</h2>")
        elif l.startswith("# "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h1>{l[2:]}</h1>")
        elif l.startswith("- "):
            if not en_lista:
                html.append("<ul>"); en_lista = True
            html.append(f"<li>{l[2:]}</li>")
        elif l.strip() == "":
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append("")
        else:
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<p>{l}</p>")
    if en_lista:
        html.append("</ul>")
    texto_html = "\n".join(html)
    texto_html = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", texto_html)
    texto_html = re.sub(r"`(.+?)`", r"<code>\1</code>", texto_html)
    return texto_html

nb_en_disco = nbformat.read(NOTEBOOK_PATH, as_version=4)

partes_html = []
nombres_incrustados = set()

for cell in nb_en_disco.cells:
    if cell.cell_type == "markdown":
        partes_html.append(markdown_a_html(cell.source))
    elif cell.cell_type == "code":
        for nombre in re.findall(r'figuras_reporte_vdem\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in figuras_por_nombre:
                partes_html.append(f"<h4>Figura: {nombre}</h4>")
                partes_html.append(figuras_por_nombre[nombre].to_html(full_html=False, include_plotlyjs="cdn"))
                nombres_incrustados.add(("figura", nombre))
        for nombre in re.findall(r'tablas_reporte_vdem\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in tablas_por_nombre:
                partes_html.append(f"<h4>Tabla: {nombre}</h4>")
                partes_html.append(tablas_por_nombre[nombre].to_html())
                nombres_incrustados.add(("tabla", nombre))

html_path = OUT_DIR / "reporte_disponibilidad_vdem.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Reporte de disponibilidad V-Dem</title></head><body>\n")
    f.write("\n".join(partes_html))
    f.write("\n</body></html>")

figuras_faltantes = set(figuras_por_nombre) - {n for t, n in nombres_incrustados if t == "figura"}
tablas_faltantes = set(tablas_por_nombre) - {n for t, n in nombres_incrustados if t == "tabla"}

print(f"Exportado: {html_path}")
print(f"Figuras incrustadas: {len(figuras_por_nombre) - len(figuras_faltantes)} / {len(figuras_por_nombre)}")
print(f"Tablas incrustadas: {len(tablas_por_nombre) - len(tablas_faltantes)} / {len(tablas_por_nombre)}")
if figuras_faltantes:
    print(f"⚠️ Figuras en memoria pero no encontradas en el .ipynb guardado: {figuras_faltantes}")
if tablas_faltantes:
    print(f"⚠️ Tablas en memoria pero no encontradas en el .ipynb guardado: {tablas_faltantes}")


Exportado: data\processed\reporte_disponibilidad_vdem.html
Figuras incrustadas: 11 / 11
Tablas incrustadas: 2 / 2
